#**GeoMine Sentinel** : Guarding land, law, and life from above.
---
GeoMine Sentinel: An AI-powered satellite monitoring system that detects mining footprints and cross-verifies them against legal and geospatial records to flag potential unauthorized mining.

'while the world looks up to clean energy as the uncompromisable future, througout the regions of devloping/underdevloped nations prevail the harvestation of energy at the cost of human life, environment and human future. The practice of unauthorized mining is still relevent in few regions far from ordinary livelyhood, we solve the problem by building a system that identify unauthorized mining footprint.

##importing necessary libraries

In [ ]:
!pip install torch transformers datasets evaluate accelerate geopandas rasterio earthengine-api geemap

In [ ]:
import torch
from huggingface_hub import login, whoami
from transformers import (
    SegformerImageProcessor, #AutoImageProcessor
    SegformerForImageClassification, #AutoModel
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    pipeline
)
from huggingface_hub import login, whoami
#from evaluate import load
from datasets import load_dataset
from PIL import Image
import numpy as np, pandas as pd, matplotlib.pyplot as plt

##login()

In [ ]:
login()

In [ ]:
whoami()

In [ ]:
device=torch.device('cuda'if torch.device.is_cuda() else 'cpu')

## Dataset: [Global mining footprint mapped from high-resolution satellite imagery by Zenodo](https://zenodo.org/records/6806817)
---
The dataset consists of mining polygons


1. Mining polygons: the existing dataset consists of the location of the mining footprints
2. Satellite Imagery: We must use the location to download the mining footprints from satellite imagery, preferably from Sentinel-2 or Google Earth
3. Combining: We then combine the polygons with the satellite images.
4. We feed the downloaded images to a segmentation model and compare its prediction to the polygons



In [8]:
import os
import geopandas as gpd

In [10]:
shp_path=r'/content/drive/MyDrive/Supplementary 1：mine area polygons/74548 mine polygons/74548_projected.shp'
mines=gpd.read_file(shp_path)

In [11]:
print(f'''
No. of polygons : {len(mines)}
Coordinate System : {mines.crs}
Columns : {mines.columns.tolist()}
{mines.head()}''')


No. of polygons : 74548
Coordinate System : PROJCS["WGS 84 / NSIDC EASE-Grid Global",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Columns : ['OBJECTID', 'Name', 'Shape_Le_1', 'Shape_Area', 'geometry']
   OBJECTID Name    Shape_Le_1    Shape_Area  \
0         1   Au    507.731122  1.213696e+04   
1         2   Au  22248.346547  6.566996e+06   
2         3   Au  19467.203319  7.109258e+06   
3         4   Fe   8281.186849  2.323061e+06   
4         5    0   1492.049584  1.231474e+05   

                                            geometry  
0  POLYGON Z ((9746741.997 3480087.392 0, 9746759...  
1  POLYGON Z

In [14]:
import ee #Google Earth Engine will be used apply filter to Sentinal-2 Data
ee.Authenticate() #authenticating EarthEngine

In [19]:
ee.Initialize(project='mining-footprint')
print(ee.String("Connected!").getInfo())

EEException: Google Earth Engine API has not been used in project mining-footprint before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/earthengine.googleapis.com/overview?project=mining-footprint then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry.

##Preprocessing & Augmentation

In [ ]:
processor = SegformerImageProcessor.from_pretrained('nvidia/mit-b0')
print(processor)